# Track C — 03. Signatures Cleaning

Collapse the per-sequencing-run genomic-instability signatures table to one row per model.
Filters to the default sequencing entry, keeps only the evidence columns, and casts `wgd`
to a nullable boolean. Writes to `Track - C/outputs/signatures/`.

In [1]:
import os
import pandas as pd
from data_utils import PARQUET_CLEAN, REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
OUT_DIR      = os.path.join(PROJECT_ROOT, 'Track - C', 'outputs', 'signatures')
os.makedirs(OUT_DIR, exist_ok=True)

KEEP_COLS = ['model_id', 'msiscore', 'lohfraction', 'wgd', 'cin', 'ploidy', 'aneuploidy']
print('output dir:', OUT_DIR)


output dir: C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\signatures


## 1. Filter to default entry, keep evidence columns, collapse to model grain

In [2]:
df = pd.read_parquet(os.path.join(PARQUET_CLEAN, 'signatures_clean.parquet'))

df = df[df['isdefaultentryformodel'] == 'yes'].copy()
print(f'After default-entry filter: {len(df)} rows, {df["modelid"].nunique()} models')

df['modelid'] = df['modelid'].str.upper()
df = df.rename(columns={'modelid': 'model_id'})

signatures_clean = df[KEEP_COLS].copy()
signatures_clean['wgd'] = signatures_clean['wgd'].astype('boolean')

print(f'\nFinal grain: {len(signatures_clean)} rows, {signatures_clean["model_id"].nunique()} unique models')
print(f'Nulls remaining:\n{signatures_clean.isnull().sum()}')


After default-entry filter: 1955 rows, 1955 models

Final grain: 1955 rows, 1955 unique models
Nulls remaining:
model_id         0
msiscore         0
lohfraction    333
wgd            333
cin            333
ploidy         333
aneuploidy     333
dtype: int64


## 2. Write output

In [3]:
OUT_PATH = os.path.join(OUT_DIR, 'signatures_model_level.parquet')
signatures_clean.to_parquet(OUT_PATH)
print(f'Saved to {OUT_PATH}')


Saved to C:\Disertation\UoB-GeneTraceAI-25-26\Track - C\outputs\signatures\signatures_model_level.parquet
